# Subsystem 2 — Social Trend Signal Integration

## Objective

This updated notebook aligns **Subsystem 2** with the current project workflow.

Subsystem 2 is responsible for:

- extracting **social trend signals** from Twitter data
- converting discovered topics into a **trend score**
- mapping social trends to **inventory categories/products**
- integrating the social signal with **Subsystem 1 output**
- exporting an **integrated recommendation dataset** for downstream risk scoring or dashboard use

### Position in the overall workflow

```text
Historical sales data
    -> Rolling window builder
    -> Subsystem 1 (category growth prediction + product allocation)
    -> product_recommendations.csv

Twitter / external social data
    -> Subsystem 2 (trend discovery + trend scoring)
    -> social_trend_signals.csv

Integration
    -> merge Subsystem 1 output with Subsystem 2 signal
    -> integrated_recommendations.csv
    -> dashboard
```


## 1. Load external social dataset

This notebook keeps your teammate's original idea of using Twitter data as an external signal.


In [6]:
import kagglehub
import pandas as pd
from pathlib import Path

# Download dataset from Kaggle
dataset_path = kagglehub.dataset_download(
    "mcantoni81/twitter-dataset-the-intel-raptor-release"
)

print("Dataset downloaded to:", dataset_path)

# Find the CSV inside the downloaded folder
dataset_path = Path(dataset_path)

csv_files = list(dataset_path.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found in downloaded dataset")

tweet_path = csv_files[0]
print("Using file:", tweet_path)

df = pd.read_csv(tweet_path)

print("Dataset shape:", df.shape)
df.head()

Dataset downloaded to: /Users/ayemyatmyathmwe/.cache/kagglehub/datasets/mcantoni81/twitter-dataset-the-intel-raptor-release/versions/1
Using file: /Users/ayemyatmyathmwe/.cache/kagglehub/datasets/mcantoni81/twitter-dataset-the-intel-raptor-release/versions/1/intel.csv
Dataset shape: (10646, 12)


,Unnamed: 0,author_id,author_name,author_username,created_at,edit_history_tweet_ids,id,lang,public_metrics,text,retweet_count,like_count
0,0,499672969,fixitfixitfixit,fixitfixitfixit,2022-10-26 23:58:48+00:00,['1585420714684940288'],1585420714684940288,en,"{'retweet_count': 0, 'reply_count': 0, 'like_c...",(US)Intel Core i9-13900K Desktop Processor - O...,0,0
1,1,499672969,fixitfixitfixit,fixitfixitfixit,2022-10-26 23:38:26+00:00,['1585415588205236232'],1585415588205236232,en,"{'retweet_count': 0, 'reply_count': 0, 'like_c...",(US)Intel Core i9-13900K Desktop Processor - O...,0,0
2,2,499672969,fixitfixitfixit,fixitfixitfixit,2022-10-26 23:18:26+00:00,['1585410554197311488'],1585410554197311488,en,"{'retweet_count': 0, 'reply_count': 0, 'like_c...",(US)Intel Core i9-13900K Desktop Processor - O...,0,0
3,3,499672969,fixitfixitfixit,fixitfixitfixit,2022-10-26 22:58:24+00:00,['1585405512463646720'],1585405512463646720,en,"{'retweet_count': 0, 'reply_count': 0, 'like_c...",(US)Intel Core i9-13900K Desktop Processor - O...,0,0
4,4,499672969,fixitfixitfixit,fixitfixitfixit,2022-10-26 22:38:22+00:00,['1585400472713760768'],1585400472713760768,en,"{'retweet_count': 0, 'reply_count': 0, 'like_c...",(US)Intel Core i9-13900K Desktop Processor - O...,0,0


## 2. Text preprocessing

Tweets are cleaned before embedding/clustering to reduce noise.


In [7]:
import kagglehub
import pandas as pd
from pathlib import Path
import re

def clean_for_bert(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # URLs
    text = re.sub(r"@\w+", " ", text)                        # mentions
    text = re.sub(r"#", " ", text)                            # keep hashtag word, remove '#'
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text)              # punctuation
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

df["clean_text"] = df["text"].apply(clean_for_bert)
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

print("Rows after cleaning / dedup:", len(df))
df[["text", "clean_text"]].head()


Rows after cleaning / dedup: 5659


,text,clean_text
0,(US)Intel Core i9-13900K Desktop Processor - O...,us intel core i9 13900k desktop processor othe...
1,(US)Intel Core i9-13900K Desktop Processor - O...,us intel core i9 13900k desktop processor othe...
2,(US)Intel Core i9-13900K Desktop Processor - O...,us intel core i9 13900k desktop processor othe...
3,(US)Intel Core i9-13900K Desktop Processor - O...,us intel core i9 13900k desktop processor othe...
4,(US)Intel Core i9-13900K Desktop Processor - O...,us intel core i9 13900k desktop processor othe...


## 3. Semantic embedding and topic discovery

These steps remain part of Subsystem 2 because they convert raw social text into structured trend topics.


In [8]:
# Install only if needed in a fresh environment
# !pip install -U sentence-transformers hdbscan umap-learn

from sentence_transformers import SentenceTransformer
import hdbscan

texts = df["clean_text"].astype(str).tolist()

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=1,
    metric="euclidean"
)
labels = clusterer.fit_predict(embeddings)
df["topic_id"] = labels

pd.Series(labels).value_counts().head(10)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/89 [00:00<?, ?it/s]

-1     3899
 18     116
 7      110
 20      80
 23      59
 39      51
 49      49
 21      48
 30      45
 10      45
Name: count, dtype: int64

## 4. Topic interpretation with TF-IDF keywords

We convert each discovered cluster into a keyword representation.  
These keywords will later be used for **category / product matching** during integration.


In [11]:
from collections import defaultdict
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

df_topics = df[df["topic_id"] != -1].copy()

topic_docs = (
    df_topics.groupby("topic_id")["clean_text"]
    .apply(lambda s: " ".join(s.astype(str)))
    .reset_index()
)

vectorizer = CountVectorizer(stop_words="english", min_df=1)
X_counts = vectorizer.fit_transform(topic_docs["clean_text"])
tfidf = TfidfTransformer()
X_tfidf = tfidf.fit_transform(X_counts)
terms = np.array(vectorizer.get_feature_names_out())

def top_words_per_topic(X_tfidf, topic_ids, terms, top_n=10):
    rows = []
    for row_i, topic in enumerate(topic_ids):
        row = X_tfidf[row_i].toarray().ravel()
        top_idx = row.argsort()[::-1][:top_n]
        keywords = terms[top_idx].tolist()
        rows.append({
            "topic_id": topic,
            "topic_keywords": keywords,
            "topic_keyword_text": " ".join(keywords)
        })
    return pd.DataFrame(rows)

topic_keywords_df = top_words_per_topic(
    X_tfidf=X_tfidf,
    topic_ids=topic_docs["topic_id"].tolist(),
    terms=terms,
    top_n=10
)

topic_keywords_df.head()


,topic_id,topic_keywords,topic_keyword_text
0,0,"[realsense, d435, vtuber, camera, depth, 2022,...",realsense d435 vtuber camera depth 2022 10 int...
1,1,"[amp, exposetheperp, targetedindividuals, pain...",amp exposetheperp targetedindividuals pain ind...
2,2,"[china, chip, ceo, restrictions, inevitable, c...",china chip ceo restrictions inevitable calls i...
3,3,"[finals, intelafricamasters, north, tune, regi...",finals intelafricamasters north tune regional ...
4,4,"[graph, neural, training, research, optimizing...",graph neural training research optimizing scal...


## 5. Daily trend scoring

The teammate's trend-scoring logic is kept, but this notebook now pushes the result toward integration.


In [12]:
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
df = df.dropna(subset=["created_at"]).copy()
df["period"] = df["created_at"].dt.date

daily_mentions = (
    df[df["topic_id"] != -1]
      .groupby(["topic_id", "period"])
      .size()
      .reset_index(name="mentions_this_period")
      .sort_values(["topic_id", "period"])
)

daily_mentions["mentions_last_period"] = (
    daily_mentions.groupby("topic_id")["mentions_this_period"].shift(1)
)

m_last = daily_mentions["mentions_last_period"]
daily_mentions["volume_growth"] = np.where(
    (m_last.isna()) | (m_last <= 0),
    0.0,
    (daily_mentions["mentions_this_period"] - m_last) / m_last
)

daily_mentions["novelty_score"] = daily_mentions["mentions_last_period"].isna().astype(int)

if {"retweet_count", "like_count"}.issubset(df.columns):
    daily_engagement = (
        df[df["topic_id"] != -1]
          .groupby(["topic_id", "period"])
          .apply(lambda g: (g["retweet_count"].fillna(0) + g["like_count"].fillna(0)).mean())
          .reset_index(name="engagement_score")
    )
    daily_mentions = daily_mentions.merge(daily_engagement, on=["topic_id", "period"], how="left")
else:
    daily_mentions["engagement_score"] = 0.0

# Normalize and combine
W1, W2, W3 = 0.6, 0.3, 0.1
daily_mentions["volume_growth_norm"] = np.tanh(daily_mentions["volume_growth"])

max_eng = daily_mentions["engagement_score"].max()
if pd.isna(max_eng) or max_eng <= 0:
    daily_mentions["engagement_score_norm"] = 0.0
else:
    daily_mentions["engagement_score_norm"] = daily_mentions["engagement_score"] / max_eng

daily_mentions["trend_score"] = (
    W1 * daily_mentions["volume_growth_norm"] +
    W2 * daily_mentions["engagement_score_norm"] +
    W3 * daily_mentions["novelty_score"]
).clip(0, 1)

daily_mentions["is_trending"] = (daily_mentions["trend_score"] >= 0.70).astype(int)

latest_period = daily_mentions["period"].max()
latest_topic_trends = (
    daily_mentions[daily_mentions["period"] == latest_period]
    .merge(topic_keywords_df, on="topic_id", how="left")
    .sort_values("trend_score", ascending=False)
    .reset_index(drop=True)
)

latest_topic_trends.head(10)


,topic_id,period,mentions_this_period,mentions_last_period,volume_growth,novelty_score,engagement_score,volume_growth_norm,engagement_score_norm,trend_score,is_trending,topic_keywords,topic_keyword_text
0,5,2022-10-26,20,3.0,5.666667,0,0.550000,0.999976,0.002717,0.600801,0,"[mobileye, ipo, unit, driving, intel, autonomo...",mobileye ipo unit driving intel autonomous mbl...
1,17,2022-10-26,26,4.0,5.500000,0,0.000000,0.999967,0.000000,0.599980,0,"[seller, earn, associate, purchases, qualifyin...",seller earn associate purchases qualifying atc...
2,37,2022-10-26,7,2.0,2.500000,0,0.000000,0.986614,0.000000,0.591969,0,"[box, amazon, 13, cpu, intel, bx8071513900k, 2...",box amazon 13 cpu intel bx8071513900k 22 core ...
3,40,2022-10-26,10,4.0,1.500000,0,1.300000,0.905148,0.006422,0.545015,0,"[zen4, 5800x3d, ryzen, amd, lake, raptor, 7000...",zen4 5800x3d ryzen amd lake raptor 7000 5000 a...
4,27,2022-10-26,2,1.0,1.000000,0,26.500000,0.761594,0.130900,0.496227,0,"[zen, beater, review, amd, 13900k, i9, core, i...",zen beater review amd 13900k i9 core intel ver...
5,39,2022-10-26,2,1.0,1.000000,0,3.500000,0.761594,0.017289,0.462143,0,"[7950x, ryzen, vs, 13900k, i9, amd, core, inte...",7950x ryzen vs 13900k i9 amd core intel faster...
6,49,2022-10-26,8,4.0,1.000000,0,1.000000,0.761594,0.004940,0.458438,0,"[13900k, i9, core, intel, cpu, 13, pc, i5, 136...",13900k i9 core intel cpu 13 pc i5 13600k 117
7,6,2022-10-26,6,3.0,1.000000,0,0.333333,0.761594,0.001647,0.457450,0,"[laptop, acer, lenovo, hp, notebook, ssd, dell...",laptop acer lenovo hp notebook ssd dell intel ...
8,41,2022-10-26,5,3.0,0.666667,0,0.600000,0.582783,0.002964,0.350559,0,"[rtx4090, 13900k, i9, 4090, rtx, core, pc, cpu...",rtx4090 13900k i9 4090 rtx core pc cpu 1000w 7...
9,7,2022-10-26,21,13.0,0.615385,0,0.000000,0.547906,0.000000,0.328744,0,"[qcom, intc, layoffs, layoff, jobs, hightech, ...",qcom intc layoffs layoff jobs hightech thelayo...


## 6. Export raw social trend signal

This file is useful for auditability and report evidence.


In [13]:
latest_topic_trends.to_csv("social_trend_signals.csv", index=False)
print("Saved: social_trend_signals.csv")


Saved: social_trend_signals.csv


## 7. Load Subsystem 1 output for integration

Subsystem 1 already produces the product recommendation table.  
This notebook now uses that output as the main input for integration.

Expected columns from Subsystem 1 include:

- `CategoryName`
- `ProductName`
- `product_share`
- `product_qty`
- `category_qty`
- `probability_up`
- `growth_signal`
- `category_adjusted_forecast`
- `recommended_qty`


In [14]:
def load_subsystem1_output():
    candidate_files = [
        Path("subsystem1_product_recommendations.csv"),
        Path("alloc_out.csv"),
        Path("/mnt/data/product_recommendations.csv"),
        Path("/mnt/data/alloc_out.csv"),
    ]
    for p in candidate_files:
        if p.exists():
            return pd.read_csv(p), p
    raise FileNotFoundError(
        "Subsystem 1 output not found. Save alloc_out as product_recommendations.csv first."
    )

sub1_df, sub1_path = load_subsystem1_output()
print("Loaded Subsystem 1 output from:", sub1_path)
print(sub1_df.shape)
sub1_df.head()


Loaded Subsystem 1 output from: subsystem1_product_recommendations.csv
(21, 11)


,CategoryName,ProductName,product_share,product_qty,category_qty,probability_up,growth_signal,category_adjusted_forecast,base_reorder_qty,confidence_score,forecast_score
0,Storage,Crucial CT525MX300SSD1,0.088795,42,473,0.848593,1.0,170.915587,15,0.848593,0.481186
1,Storage,Corsair Dominator Platinum,0.164905,78,473,0.848593,1.0,170.915587,28,0.848593,0.530321
2,Storage,Hitachi HUA723020ALA640,0.209302,99,473,0.848593,1.0,170.915587,36,0.848593,0.560070
3,Storage,PNY SSD9SC240GMDA-RB,0.247357,117,473,0.848593,1.0,170.915587,42,0.848593,0.583333
4,Storage,Samsung MZ-V6P2T0BW,0.289641,137,473,0.848593,1.0,170.915587,50,0.848593,0.612659


## 8. Build a topic-to-category bridge

Because social topics do not naturally contain inventory category IDs, we create a lightweight bridge.

### Bridge logic used here
1. tokenize each category / product name from Subsystem 1
2. tokenize each topic keyword list
3. measure keyword overlap
4. assign the best matching topic to each category
5. use the latest topic trend score as the category-level social trend signal

This rule-based bridge is transparent and easy to justify in the report.


In [15]:
STOP = {
    "intel","amd","asus","asrock","msi","gigabyte","supermicro","corsair","crucial",
    "oem","tray","gaming","edition","series","gb","tb","ddr3","ddr4","ddr5",
    "x","v","ws","pro","plus"
}

def tokenize_name(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [t for t in text.split() if len(t) > 2 and t not in STOP and not t.isdigit()]
    return set(tokens)

# category/product vocabulary
category_vocab = (
    sub1_df.groupby("CategoryName")["ProductName"]
    .apply(lambda s: set().union(*[tokenize_name(x) for x in s.astype(str)]))
    .to_dict()
)

# add category name tokens themselves
for cat in sub1_df["CategoryName"].dropna().unique():
    category_vocab[cat] = category_vocab.get(cat, set()).union(tokenize_name(cat))

topic_bridge = []
for _, row in latest_topic_trends.iterrows():
    topic_tokens = set(row.get("topic_keywords", [])) if isinstance(row.get("topic_keywords", []), list) else set(str(row.get("topic_keyword_text", "")).split())
    if not topic_tokens:
        continue
    best_cat = None
    best_overlap = 0
    for cat, cat_tokens in category_vocab.items():
        overlap = len(topic_tokens.intersection(cat_tokens))
        if overlap > best_overlap:
            best_overlap = overlap
            best_cat = cat
    topic_bridge.append({
        "topic_id": row["topic_id"],
        "matched_category": best_cat,
        "keyword_overlap": best_overlap,
        "trend_score": row["trend_score"],
        "is_trending": row["is_trending"],
        "topic_keyword_text": row.get("topic_keyword_text", "")
    })

topic_bridge_df = pd.DataFrame(topic_bridge)
topic_bridge_df.head(10)


,topic_id,matched_category,keyword_overlap,trend_score,is_trending,topic_keyword_text
0,5,NaN,0,0.600801,0,mobileye ipo unit driving intel autonomous mbl...
1,17,NaN,0,0.599980,0,seller earn associate purchases qualifying atc...
2,37,CPU,2,0.591969,0,box amazon 13 cpu intel bx8071513900k 22 core ...
3,40,NaN,0,0.545015,0,zen4 5800x3d ryzen amd lake raptor 7000 5000 a...
4,27,CPU,1,0.496227,0,zen beater review amd 13900k i9 core intel ver...
5,39,CPU,1,0.462143,0,7950x ryzen vs 13900k i9 amd core intel faster...
6,49,CPU,2,0.458438,0,13900k i9 core intel cpu 13 pc i5 13600k 117
7,6,NaN,0,0.457450,0,laptop acer lenovo hp notebook ssd dell intel ...
8,41,CPU,2,0.350559,0,rtx4090 13900k i9 4090 rtx core pc cpu 1000w 7...
9,7,NaN,0,0.328744,0,qcom intc layoffs layoff jobs hightech thelayo...


## 9. Category-level social signal

For each category, keep the strongest matched topic from the latest period.


In [16]:
category_social = (
    topic_bridge_df.dropna(subset=["matched_category"])
    .sort_values(["matched_category", "trend_score"], ascending=[True, False])
    .drop_duplicates(subset=["matched_category"])
    .rename(columns={"matched_category": "CategoryName"})
    [["CategoryName", "trend_score", "is_trending", "topic_keyword_text", "keyword_overlap"]]
    .rename(columns={
        "trend_score": "social_trend_score",
        "is_trending": "social_is_trending"
    })
)

# optional fallback if nothing matched
if category_social.empty:
    category_social = pd.DataFrame({
        "CategoryName": sorted(sub1_df["CategoryName"].dropna().unique()),
        "social_trend_score": 0.0,
        "social_is_trending": 0,
        "topic_keyword_text": "",
        "keyword_overlap": 0
    })

category_social.head()



,CategoryName,social_trend_score,social_is_trending,topic_keyword_text,keyword_overlap
2,CPU,0.591969,0,box amazon 13 cpu intel bx8071513900k 22 core ...,2


## Export Category Social CSV 

In [17]:
category_social.to_csv("category_social_trends.csv", index=False)

## 13. Notes for the report

### workflow
- Subsystem 1 remains the **core inventory recommender** based on historical demand
- Subsystem 2 now acts as an **external signal enrichment layer**
- The final dashboard reads a **single integrated output**


### Suggested architecture wording
> Subsystem 1 generates category-driven product recommendations from historical demand patterns.  
> Subsystem 2 extracts external social trend signals from Twitter data using semantic embedding, topic clustering, and daily trend scoring.  
> The two outputs are integrated at category level through a transparent bridge and weighted fusion rule to produce the final integrated recommendation dataset used by the dashboard.
